<a href="https://colab.research.google.com/github/Dana-El/CCI-HW5/blob/main/Lab0_Search_Spectrum_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab0_Search_Spectrum_Student.ipynb)


# Lab 0 The Search Spectrum — From Exact Match to RAG
## CCI Session 6

**Duration:** 30 minutes

### Clinical Scenario
> A KHCC clinician searches through oncology patient notes to answer clinical questions.
> We progress through 7 retrieval methods — each one solves a limitation of the previous.

### Objectives
- Implement **exact**, **regex**, and **fuzzy** text search
- Build **TF-IDF** and **BM25** lexical retrieval indices
- Create a **semantic search** engine with free sentence embeddings
- Assemble an **oversimplified RAG** pipeline (retrieve → augment → generate)

### The Search Spectrum
```
Exact ──► Regex ──► Fuzzy ──► TF-IDF ──► BM25 ──► Semantic ──► RAG
◄── more literal                                    more semantic ──►
◄── faster                                               slower ──►
◄── no model needed                             model required ──►
```


In [3]:
!pip install -q rapidfuzz rank_bm25 scikit-learn sentence-transformers openai pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.6 MB/s eta 0:00:00


In [4]:
# Synthetic KHCC oncology notes (15 notes)
corpus = [
    'Patient A, 4yo male. Wilms tumor (nephroblastoma) stage II favorable histology. '
    'Vincristine 1.5 mg/m2 IV and actinomycin-D per COG AREN0532. Post-nephrectomy week 1.',
    'Patient B, 7yo female. B-cell precursor ALL. Induction: dexamethasone, vincristine, '
    'PEG-asparaginase, doxorubicin. Bone marrow 85% blasts at diagnosis.',
    'Patient C, 12yo male. Hodgkin lymphoma stage IIB. ABVD: adriamycin 25 mg/m2, '
    'bleomycin, vinblastine, dacarbazine. Biopsy confirmed Reed-Sternberg cells.',
    'Patient D, 5yo female. Medulloblastoma post-resection. Craniospinal irradiation '
    '36 Gy with concurrent carboplatin and vincristine.',
    'Patient E, 9yo male. Relapsed bilateral Wilms tumor. Salvage ICE: carboplatin, '
    'etoposide, ifosfamide. Stem cell transplant planned if second CR achieved.',
    'Patient F, 3yo female. Neuroblastoma stage IV, MYCN amplified. High-risk induction: '
    'cisplatin, etoposide, doxorubicin, cyclophosphamide, vincristine (GPOH NB2004).',
    'Patient G, 14yo male. Osteosarcoma distal femur. MAP: methotrexate 12 g/m2, '
    'adriamycin 75 mg/m2, cisplatin 120 mg/m2. Pre-operative cycle 2.',
    'Patient H, 6yo female. AML M4. Induction: cytarabine 100 mg/m2 and daunorubicin '
    '50 mg/m2 (7+3). Lumbar puncture negative for CNS disease.',
    'Patient I, 10yo male. Ewing sarcoma pelvis. VDC/IE: vincristine, doxorubicin, '
    'cyclophosphamide alternating with ifosfamide and etoposide.',
    'Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrectomy. '
    'Adjuvant vincristine and actinomycin-D per COG AREN0532.',
    'Patient K, 11yo male. T-cell ALL relapse. Nelarabine-based salvage. Intrathecal '
    'methotrexate CNS prophylaxis. MRD monitoring every 2 weeks.',
    'Patient L, 2yo female. Hepatoblastoma PRETEXT III. PLADO: cisplatin 80 mg/m2 and '
    'doxorubicin 60 mg/m2. Surgical resection after 4 cycles.',
    'Patient M, 15yo female. Embryonal rhabdomyosarcoma. VAC/VI: vincristine, '
    'actinomycin-D, cyclophosphamide alternating with vincristine and irinotecan.',
    'Patient N, 7yo male. Burkitt lymphoma stage IV, bone marrow involved. CODOX-M/IVAC. '
    'Tumor lysis syndrome prophylaxis: allopurinol and aggressive hydration.',
    'Patient O, 5yo male. Incidental nephrogenic rest on imaging. No active Wilms tumor. '
    'Surveillance ultrasound every 3 months per COG. No chemotherapy indicated.',
]

print(f'Corpus: {len(corpus)} clinical notes\n')
for i, note in enumerate(corpus):
    print(f'[{i:2d}] {note[:90]}...')


Corpus: 15 clinical notes

[ 0] Patient A, 4yo male. Wilms tumor (nephroblastoma) stage II favorable histology. Vincristin...
[ 1] Patient B, 7yo female. B-cell precursor ALL. Induction: dexamethasone, vincristine, PEG-as...
[ 2] Patient C, 12yo male. Hodgkin lymphoma stage IIB. ABVD: adriamycin 25 mg/m2, bleomycin, vi...
[ 3] Patient D, 5yo female. Medulloblastoma post-resection. Craniospinal irradiation 36 Gy with...
[ 4] Patient E, 9yo male. Relapsed bilateral Wilms tumor. Salvage ICE: carboplatin, etoposide, ...
[ 5] Patient F, 3yo female. Neuroblastoma stage IV, MYCN amplified. High-risk induction: cispla...
[ 6] Patient G, 14yo male. Osteosarcoma distal femur. MAP: methotrexate 12 g/m2, adriamycin 75 ...
[ 7] Patient H, 6yo female. AML M4. Induction: cytarabine 100 mg/m2 and daunorubicin 50 mg/m2 (...
[ 8] Patient I, 10yo male. Ewing sarcoma pelvis. VDC/IE: vincristine, doxorubicin, cyclophospha...
[ 9] Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrec

---
## Section 1 — Exact Text Match

The simplest possible search: is the query string present verbatim in the document?

**Strengths:** instant, zero dependencies, deterministic
**Failures:** misses `'Wilms tumour'` (British spelling), `'nephroblastoma'` (synonym), `'vincristne'` (typo)


In [5]:
def exact_search(query, docs, case_sensitive=False):
    hits = []
    for i, doc in enumerate(docs):
        if case_sensitive:
            if query in doc:
                hits.append(i)
        else:
            if query.lower() in doc.lower():
                hits.append(i)
    return hits

for q in ['Wilms tumor', 'vincristine', 'nephroblastoma', 'kidney cancer', 'Wilms tumour']:
    hits = exact_search(q, corpus)
    print(f"'{q}': {len(hits)} hits  → {hits}")

'Wilms tumor': 4 hits  → [0, 4, 9, 14]
'vincristine': 7 hits  → [0, 1, 3, 5, 8, 9, 12]
'nephroblastoma': 1 hits  → [0]
'kidney cancer': 0 hits  → []
'Wilms tumour': 0 hits  → []


---
## Section 2 — Regex (Regular Expressions)

Regex matches **patterns** rather than fixed strings. Essential for clinical text: drug dosages, staging, ICD codes.

**Strengths:** handles format variation, very fast
**Failures:** requires hand-crafted patterns; can't handle synonyms or typos


In [6]:
import re

def regex_search(pattern, docs, flags=re.IGNORECASE):
    hits = []
    for i, doc in enumerate(docs):
        match = re.search(pattern, doc, flags)
        if match:
            hits.append((i, match.group(0)))
    return hits

# Drug + dosage — pattern: vincristine followed by a number and 'mg'
print('--- vincristine dosage ---')
for hit_i, match in regex_search(r'vincristine\s+[\d.]+\s*mg', corpus):
    print(f'  [{hit_i}] matched: {match!r}')

# Any dosage unit
print('\n--- Any dosage ---')
for hit_i, match in regex_search(r'\d+\.?\d*\s*(?:mg/m2|g/m2|mg|Gy)', corpus)[:8]:
    print(f'  [{hit_i}] {match!r}')

# TODO: write your own regex to find all stage mentions (e.g. 'stage IV', 'stage IIB')
print('\n--- Staging patterns ---')
for hit_i, match in regex_search(r'stage\s+[IVX]+[AB]?', corpus):
    print(f'  [{hit_i}] {match!r}')

--- vincristine dosage ---
  [0] matched: 'Vincristine 1.5 mg'

--- Any dosage ---
  [0] '1.5 mg/m2'
  [2] '25 mg/m2'
  [3] '36 Gy'
  [6] '12 g/m2'
  [7] '100 mg/m2'
  [11] '80 mg/m2'

--- Staging patterns ---
  [0] 'stage II'
  [2] 'stage IIB'
  [5] 'stage IV'
  [9] 'stage I'
  [13] 'stage IV'


---
## Section 3 — Fuzzy Text Match

Fuzzy matching tolerates **typos and spelling variants** via edit-distance scoring.
`rapidfuzz.fuzz.partial_ratio` scores based on the best-matching substring.

**Strengths:** handles OCR errors, British vs. American spelling
**Failures:** purely character-level — `'nephroblastoma'` ≠ `'kidney cancer'` (different words, not a typo)


In [7]:
from rapidfuzz import fuzz

def fuzzy_search(query, docs, threshold=75):
    hits = []
    for i, doc in enumerate(docs):
        score = fuzz.partial_ratio(query.lower(), doc.lower())
        if score >= threshold:
            hits.append((i, score))
    hits.sort(key=lambda x: x[1], reverse=True)
    return hits

for q in ['Wilms tumour', 'vincristne', 'nephroblstoma', 'leukaemia', 'kidney cancer']:
    hits = fuzzy_search(q, corpus)
    top = [(i, s) for i, s in hits[:2]]
    print(f"'{q}': {len(hits)} hits, top: {top}")

'Wilms tumour': 4 hits, top: [(0, 91.66666666666666), (4, 91.66666666666666)]
'vincristne': 7 hits, top: [(0, 90.0), (1, 90.0)]
'nephroblstoma': 3 hits, top: [(0, 92.3076923076923), (5, 84.61538461538461)]
'leukaemia': 0 hits, top: []
'kidney cancer': 0 hits, top: []


---
## Section 4 — TF-IDF (Term Frequency–Inverse Document Frequency)

TF-IDF maps documents to **sparse vectors**. Each dimension is a vocabulary term, weighted by:
- **TF**: how often the term appears in *this* document
- **IDF**: penalises common terms that appear in *many* documents (like "patient")

Retrieval = cosine similarity between the query vector and all document vectors.

**Strengths:** fast, no neural model needed, keyword-aware
**Failures:** `'kidney'` ≠ `'renal'`, no morphology awareness unless you add stemming


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# TODO: create a TfidfVectorizer with stop_words='english' and ngram_range=(1, 2)
# then fit_transform on the corpus
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'TF-IDF matrix: {tfidf_matrix.shape}  (docs × terms)')

def tfidf_search(query, k=5):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:k]
    return [(i, scores[i]) for i in top_indices if scores[i] > 0]

for q in ['vincristine actinomycin Wilms', 'leukemia blast cells', 'kidney cancer child']:
    print(f"\nTF-IDF: '{q}'")
    for i, s in tfidf_search(q):
        print(f'  [{i}] score={s:.4f} | {corpus[i][:80]}...')

TF-IDF matrix: (15, 376)  (docs × terms)

TF-IDF: 'vincristine actinomycin Wilms'
  [9] score=0.3144 | Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrectom...
  [12] score=0.2910 | Patient M, 15yo female. Embryonal rhabdomyosarcoma. VAC/VI: vincristine, actinom...
  [0] score=0.1751 | Patient A, 4yo male. Wilms tumor (nephroblastoma) stage II favorable histology. ...
  [14] score=0.0649 | Patient O, 5yo male. Incidental nephrogenic rest on imaging. No active Wilms tum...
  [4] score=0.0593 | Patient E, 9yo male. Relapsed bilateral Wilms tumor. Salvage ICE: carboplatin, e...

TF-IDF: 'leukemia blast cells'
  [2] score=0.1708 | Patient C, 12yo male. Hodgkin lymphoma stage IIB. ABVD: adriamycin 25 mg/m2, ble...

TF-IDF: 'kidney cancer child'


---
## Section 5 — BM25 (Best Match 25)

BM25 is the probabilistic retrieval model powering **Elasticsearch, Solr, and Lucene**.
It improves on TF-IDF with two mechanisms:

1. **TF saturation** — the 100th occurrence of a term adds far less than the 1st
2. **Document-length normalisation** — a short note that matches scores higher than a long one

**Strengths:** better ranking calibration than TF-IDF; industry default for lexical search
**Still fails:** synonyms and semantics — `'kidney cancer'` still won't find `'nephroblastoma'`


In [9]:
from rank_bm25 import BM25Okapi

# TODO: tokenise corpus (lowercase + split), build BM25Okapi index
tokenised = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenised)

def bm25_search(query, k=5):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    import numpy as np
    top_indices = np.argsort(scores)[::-1][:k]
    return [(i, scores[i]) for i in top_indices if scores[i] > 0]

# Head-to-head: TF-IDF vs BM25 on three queries
print(f"{'Query':<42} {'TF-IDF top-3':<22} BM25 top-3")
print('-' * 80)
for q in ['vincristine actinomycin Wilms', 'leukemia blast cells', 'kidney cancer child']:
    t = [i for i, _ in tfidf_search(q, k=3)]
    b = [i for i, _ in bm25_search(q, k=3)]
    print(f'{q:<42} {str(t):<22} {b}')

print('\nWilms tumor (nephroblastoma = kidney cancer) notes: [0, 4, 9, 14]')
print('Does either lexical method find them for the synonym query?')

Query                                      TF-IDF top-3           BM25 top-3
--------------------------------------------------------------------------------
vincristine actinomycin Wilms              [np.int64(9), np.int64(12), np.int64(0)] [np.int64(9), np.int64(0), np.int64(12)]
leukemia blast cells                       [np.int64(2)]          []
kidney cancer child                        []                     []

Wilms tumor (nephroblastoma = kidney cancer) notes: [0, 4, 9, 14]
Does either lexical method find them for the synonym query?


---
## Section 6 — Semantic Search (Dense Retrieval)

A neural encoder maps text to **dense vectors** where similar *meanings* land close together.
`'kidney cancer'` and `'nephroblastoma'` appear in similar contexts during pre-training, so
their embeddings end up nearby — unlike TF-IDF/BM25 which only see character overlap.

We use **`all-MiniLM-L6-v2`** — a 80 MB sentence-transformers model, free, no API key needed.

**Strengths:** synonyms, paraphrasing, conceptual similarity
**Failures:** can miss exact alphanumeric codes/IDs that BM25 nails; slower at scale


In [10]:
from sentence_transformers import SentenceTransformer

print('Loading model (first run downloads ~80 MB)...')
# TODO: load 'all-MiniLM-L6-v2' and encode the corpus
st_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = st_model.encode(corpus)
print(f'Embeddings shape: {corpus_embeddings.shape}  (docs × dims)')

Loading model (first run downloads ~80 MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (15, 384)  (docs × dims)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, k=5):
    query_embedding = st_model.encode([query])
    scores = cosine_similarity(query_embedding, corpus_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:k]
    return [(i, scores[i]) for i in top_indices]

# The key demo — synonym query that stumps lexical methods
query = 'kidney cancer child'
print(f"Query: '{query}'  (no shared words with 'nephroblastoma')\n")
print(f"Exact:    {exact_search(query, corpus)}")
print(f"BM25:     {[i for i, _ in bm25_search(query, k=3)]}")
print(f"Semantic: {[i for i, _ in semantic_search(query, k=3)]}")
print('\nWilms tumor (nephroblastoma) indices: [0, 4, 9, 14]')
print('\n--- Semantic top-5 ---')
for i, s in semantic_search(query):
    print(f'  [{i}] score={s:.4f} | {corpus[i][:85]}...')

Query: 'kidney cancer child'  (no shared words with 'nephroblastoma')

Exact:    []
BM25:     []
Semantic: [np.int64(14), np.int64(9), np.int64(13)]

Wilms tumor (nephroblastoma) indices: [0, 4, 9, 14]

--- Semantic top-5 ---
  [14] score=0.4480 | Patient O, 5yo male. Incidental nephrogenic rest on imaging. No active Wilms tumor. S...
  [9] score=0.4175 | Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrectomy. Ad...
  [13] score=0.3883 | Patient N, 7yo male. Burkitt lymphoma stage IV, bone marrow involved. CODOX-M/IVAC. T...
  [0] score=0.3773 | Patient A, 4yo male. Wilms tumor (nephroblastoma) stage II favorable histology. Vincr...
  [8] score=0.3315 | Patient I, 10yo male. Ewing sarcoma pelvis. VDC/IE: vincristine, doxorubicin, cycloph...


---
## Section 7 — Oversimplified RAG

**RAG = Retrieve → Augment → Generate**

1. **Retrieve**: find the most relevant notes (semantic search)
2. **Augment**: inject them as context in the LLM prompt
3. **Generate**: the LLM answers *grounded* in real evidence — not hallucinated

This is intentionally bare-bones. Labs 1–5 in this session add structured parsing,
evaluation, agentic re-retrieval, and knowledge graphs on top of this skeleton.


In [18]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [19]:
from openai import OpenAI
client = OpenAI()

def simple_rag(question, k=3):
    # Step 1 — Retrieve: use semantic_search to get top-k notes
    retrieved_indices = semantic_search(question, k=k)
    retrieved = [corpus[i] for i, _ in retrieved_indices]
    context = '\n\n'.join(f'[{j+1}] {doc}' for j, doc in enumerate(retrieved))

    # Step 2 — Generate: call client.chat.completions.create with gpt-4o-mini
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a clinical assistant. Answer questions based only on the provided clinical notes. Cite your sources using [1], [2], etc."},
            {"role": "user", "content": f"Notes:\n{context}\n\nQuestion: {question}"}
        ]
    )
    answer = response.choices[0].message.content
    return answer, retrieved

questions = [
    'Which patients with kidney tumours received vincristine?',
    'What salvage regimen was used for the relapsed patient?',
    'Which patients have CNS involvement or received CNS prophylaxis?',
]

for q in questions:
    print(f'\n{"="*65}')
    print(f'Q: {q}')
    answer, sources = simple_rag(q)
    print(f'A: {answer}')
    print('Sources:')
    for j, s in enumerate(sources):
        print(f'  [{j+1}] {s[:80]}...')


Q: Which patients with kidney tumours received vincristine?
A: Patient J, an 8-year-old female with Wilms tumor stage I, received vincristine as part of her treatment regimen [1]. There are no other patients with kidney tumors mentioned in the provided clinical notes who received vincristine.
Sources:
  [1] Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrectom...
  [2] Patient I, 10yo male. Ewing sarcoma pelvis. VDC/IE: vincristine, doxorubicin, cy...
  [3] Patient M, 15yo female. Embryonal rhabdomyosarcoma. VAC/VI: vincristine, actinom...

Q: What salvage regimen was used for the relapsed patient?
A: For Patient E, the salvage regimen used for the relapsed bilateral Wilms tumor was ICE, which consists of carboplatin, etoposide, and ifosfamide [2].
Sources:
  [1] Patient K, 11yo male. T-cell ALL relapse. Nelarabine-based salvage. Intrathecal ...
  [2] Patient E, 9yo male. Relapsed bilateral Wilms tumor. Salvage ICE: carboplatin, e...
  [3] Patient O, 5yo mal

---
## Summary — The Search Spectrum

Each method fixes a gap in the previous one.
In production, **hybrid search** (BM25 + semantic + a reranker) often combines their strengths.
That is exactly what modern RAG stacks like LlamaIndex (Lab 1) do under the hood.


In [20]:
import pandas as pd

rows = [
    ('Exact Match',  'No',  'No',  'No',  'No',  'Instant',    'Known IDs / accession codes'),
    ('Regex',        'No',  'No',  'No',  'No',  'Very fast',  'Dosage / date / code patterns'),
    ('Fuzzy',        'Yes', 'No',  'No',  'No',  'Slow',       'Noisy / OCR / misspelled text'),
    ('TF-IDF',       'No',  'No',  'No',  'No',  'Fast',       'Keyword bag-of-words queries'),
    ('BM25',         'No',  'No',  'No',  'No',  'Fast',       'Full-text search engine default'),
    ('Semantic',     'Partial','Yes','Yes','No',  'Medium',     'Synonym / paraphrase queries'),
    ('RAG',          'Partial','Yes','Yes','Yes', 'Slow',       'Open-ended question answering'),
]
cols = ['Method', 'Typo-tolerant', 'Synonym-aware', 'Needs model', 'Generates answer',
        'Speed (scale)', 'Best for']
df = pd.DataFrame(rows, columns=cols)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 220)
print(df.to_string(index=False))


     Method Typo-tolerant Synonym-aware Needs model Generates answer Speed (scale)                        Best for
Exact Match            No            No          No               No       Instant     Known IDs / accession codes
      Regex            No            No          No               No     Very fast   Dosage / date / code patterns
      Fuzzy           Yes            No          No               No          Slow   Noisy / OCR / misspelled text
     TF-IDF            No            No          No               No          Fast    Keyword bag-of-words queries
       BM25            No            No          No               No          Fast Full-text search engine default
   Semantic       Partial           Yes         Yes               No        Medium    Synonym / paraphrase queries
        RAG       Partial           Yes         Yes              Yes          Slow   Open-ended question answering
